In [1]:
import os
import sys
import pandas as pd
import glob
import numpy as np
from scipy.signal import savgol_filter

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.hampel_filter import apply_hampel_filter
from utils.constants import Configs
from utils.helper_functions import get_smart_retail_dataset

SMART_RETAIL_OUTLIERS_LEFT_PATH: /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Data/SmartRetail/outliers/outliers_left.csv
SMART_RETAIL_OUTLIERS_RIGHT_PATH: /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Data/SmartRetail/outliers/outliers_right.csv


/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


### Fundir braço esquerdo e braço direito lado a lado

In [2]:
df = get_smart_retail_dataset()

Loading and fusing bimanual files...


## Filtragem

### Filtro de Hampel (ou Mediana) para tentar diminuir os erros do sensor

In [23]:
sensor_cols_L = ['xs_L', 'ys_L', 'zs_L', 'xe_L', 'ye_L', 'ze_L', 'xw_L', 'yw_L', 'zw_L']
sensor_cols_R = ['xs_R', 'ys_R', 'zs_R', 'xe_R', 'ye_R', 'ze_R', 'xw_R', 'yw_R', 'zw_R']
all_sensor_cols = sensor_cols_L + sensor_cols_R

# 1. HAMPEL FILTER
for col in all_sensor_cols:
    print(f"Processing Hampel: {col} ...")
    df[col + '_clean'] = df.groupby(['sub', 'repID', 'movID'])[col].transform(
        lambda x: apply_hampel_filter(x, window_size=5)
    )

print("Done")

Processing Hampel: xs_L ...
Processing Hampel: ys_L ...
Processing Hampel: zs_L ...
Processing Hampel: xe_L ...
Processing Hampel: ye_L ...
Processing Hampel: ze_L ...
Processing Hampel: xw_L ...
Processing Hampel: yw_L ...
Processing Hampel: zw_L ...
Processing Hampel: xs_R ...
Processing Hampel: ys_R ...
Processing Hampel: zs_R ...
Processing Hampel: xe_R ...
Processing Hampel: ye_R ...
Processing Hampel: ze_R ...
Processing Hampel: xw_R ...
Processing Hampel: yw_R ...
Processing Hampel: zw_R ...
Done


### Filtro de Savitzky-Golay para suavizar o resultado final 

In [24]:
window = 11  
order = 3    

def apply_savgol(series):
    return savgol_filter(series, window_length=window, polyorder=order)

cols_to_filter = [f"{col}_clean" for col in all_sensor_cols]
for col in cols_to_filter:
    print(f"Processing Savitzky-Golay: {col} ...")
    df[f'{col}_smooth'] = df.groupby(['sub', 'repID', 'movID'])[col].transform(apply_savgol)

Processing Savitzky-Golay: xs_L_clean ...
Processing Savitzky-Golay: ys_L_clean ...
Processing Savitzky-Golay: zs_L_clean ...
Processing Savitzky-Golay: xe_L_clean ...
Processing Savitzky-Golay: ye_L_clean ...
Processing Savitzky-Golay: ze_L_clean ...
Processing Savitzky-Golay: xw_L_clean ...
Processing Savitzky-Golay: yw_L_clean ...
Processing Savitzky-Golay: zw_L_clean ...
Processing Savitzky-Golay: xs_R_clean ...
Processing Savitzky-Golay: ys_R_clean ...
Processing Savitzky-Golay: zs_R_clean ...
Processing Savitzky-Golay: xe_R_clean ...
Processing Savitzky-Golay: ye_R_clean ...
Processing Savitzky-Golay: ze_R_clean ...
Processing Savitzky-Golay: xw_R_clean ...
Processing Savitzky-Golay: yw_R_clean ...
Processing Savitzky-Golay: zw_R_clean ...


### Clean Final Dataframe

In [25]:
# As colunas cinemáticas e derivadas (Jerk) já existem com estes nomes exatos no df
kinematic_cols = ['vel_L', 'acc_L', 'k_L', 'jerk_L', 'vel_R', 'acc_R', 'k_R', 'jerk_R']

# As colunas que passaram pelo Savitzky-Golay ganharam este sufixo no teu loop
smooth_cols = [f"{col}_clean_smooth" for col in all_sensor_cols]

# Os metadados mestre da tarefa
metadata_cols_final = ['sub', 'repID', 'movID', 'movType', 'step', 'movLabel', 't']

# 1. Filtrar o DataFrame mantendo APENAS o que nos interessa
df_final = df[metadata_cols_final + smooth_cols + kinematic_cols].copy()

# 2. Renomear as colunas suavizadas de volta para os nomes limpos (ex: xs_L_clean_smooth -> xs_L)
# Mantemos a exata mesma ordem da junção acima!
df_final.columns = metadata_cols_final + all_sensor_cols + kinematic_cols

In [26]:
df.head()

,sub,repID,movID,movType,step,xs_L,ys_L,zs_L,xe_L,ye_L,...,zw_L_clean_smooth,xs_R_clean_smooth,ys_R_clean_smooth,zs_R_clean_smooth,xe_R_clean_smooth,ye_R_clean_smooth,ze_R_clean_smooth,xw_R_clean_smooth,yw_R_clean_smooth,zw_R_clean_smooth
0,1,1,1,0.0,1,2.357,-190.389,1448.24,28.459,-172.787,...,867.144769,9.058469,178.092427,1464.560070,18.683706,180.842441,1143.559371,46.037028,170.737126,882.544958
1,1,1,1,0.0,2,3.182,-192.290,1447.52,27.931,-173.406,...,866.873175,9.272993,178.004993,1464.830979,19.149049,180.823888,1143.842168,46.163147,170.434531,882.801315
2,1,1,1,0.0,3,3.622,-193.419,1446.96,27.135,-174.201,...,866.697569,9.398797,177.873200,1465.052284,19.484231,180.812455,1144.073357,46.191874,170.082583,883.013783
3,1,1,1,0.0,4,3.895,-194.151,1446.52,26.100,-174.614,...,866.598030,9.449650,177.701747,1465.232028,19.721955,180.798661,1144.261690,46.153879,169.692812,883.189668
4,1,1,1,0.0,5,4.207,-194.841,1446.10,25.032,-174.843,...,866.554639,9.439322,177.495331,1465.378252,19.894923,180.773028,1144.415921,46.079830,169.276753,883.336273


### Save to Csv

In [ ]:
cache_path = Configs.SMART_RETAIL_FILTERED_DATA_BIMANUAL
df_final.to_csv(cache_path, index=False)
print(f"Processed bimanual data saved to {cache_path}")

Processed bimanual data saved to ../../data/processed/filtered_data_bimanual.csv
